# Ask AOPWiki a question

Find Phenobarbital and read its identifiers. The model uses rdfsolve's tools to read AOPWiki; it does not receive a query to copy.

We open the included schema, not a new mining run. Only the requested records are read from the endpoint.

Install `uv pip install -e ".[agents,mcp,notebooks]"` in your notebook environment. Put `OPENAI_API_KEY` in `notebooks/.env`. `OPENAI_MODEL` is optional; this example uses GPT-5.4 mini by default.

In [ ]:
import os
import sys
from dotenv import load_dotenv
from mcp import Client as MCPClient, StdioServerParameters
from pydantic_ai import Agent
from pydantic_ai.usage import UsageLimits
from rdfsolve.pydantic_ai import mcp_tools

load_dotenv("../.env");

## Open the tools

The server reads the saved schema. It can list types and operations, find records, read fields, and follow links. The session keeps the queries and returned data.

In [ ]:
server = MCPClient(
    StdioServerParameters(
        command=sys.executable,
        args=["-m", "rdfsolve.mcp", "--schema", "../data/aopwikirdf.schema.json",
              "--source-id", "aopwikirdf", "--log", "output/session.json"],
    ),
    read_timeout_seconds=120,
)

## Ask a question

Change the question to try another name. This short run has a limit on model requests, tool calls, and tokens.

In [ ]:
async with server:
    agent = Agent(
        os.getenv("OPENAI_MODEL", "openai:gpt-5.4-mini-2026-03-17"),
        toolsets=[await mcp_tools(server)],
        instructions=server.instructions,
        model_settings={"max_tokens": 800},
        retries=1,
    )
    answer = await agent.run(
        "Find Phenobarbital. Give its chemical identifier and the other names or identifiers attached to that chemical.",
        usage_limits=UsageLimits(request_limit=8, tool_calls_limit=12, total_tokens_limit=15000),
    )
print(answer.output)

## See what it used

The table lists the tools used. Open an entry below it to see its arguments and answer, or the SPARQL queries and returned rows.

In [ ]:
from IPython.display import display
from rdfsolve.query_log import QueryLog

log = QueryLog.read("output/session.json")
display(log.tools())
display(log)